In [1]:
import torch
from tqdm.notebook import tqdm
from typing import defaultdict

In [3]:
arr = torch.randint(low=1, high=10, size=(10_000, 2)).float()

arr[..., 0] = arr[..., 0] * 100
arr[..., 1] = arr[..., 1] + torch.rand((arr.size(0)))

arr

tensor([[700.0000,   4.5164],
        [200.0000,   7.7541],
        [600.0000,   2.6500],
        ...,
        [200.0000,   5.2623],
        [300.0000,   5.0798],
        [700.0000,   7.7102]])

In [4]:
class DDN(torch.nn.Module):
    def __init__(self, *args, **kwargs) -> None:
        super().__init__(*args, **kwargs)

        self.model = torch.nn.Linear(2, 2, bias=True)

        self.scorer = torch.nn.Linear(2, 1, bias=False)

    def forward(self, x):
        return self.model(x) / x

In [5]:
DDN()

DDN(
  (model): Linear(in_features=2, out_features=2, bias=True)
  (scorer): Linear(in_features=2, out_features=1, bias=False)
)

In [7]:
test = DDN()

optimizer = torch.optim.AdamW(test.parameters(), lr=.1)

for epoch in range(1000):
    arr = torch.randint(low=1, high=10, size=(10_000, 2)).float()

    arr[..., 0] = arr[..., 0] * 100
    arr[..., 1] = arr[..., 1] + torch.rand((arr.size(0)))

    max_arr_0 = torch.max(arr, dim=0).values
    pbar = range(arr.size(0) // 1_024)
    if epoch % 100 == 0:
        pbar = tqdm(range(arr.size(0) // 1_024))
    for batch in pbar:
        optimizer.zero_grad(set_to_none=True)

        predictions = test(arr[batch * 1024: batch * 1024 + 1024].float() / max_arr_0[None, :])

        with torch.no_grad():
            targets = arr[batch * 1024: batch * 1024 + 1024, 1] / arr[batch * 1024: batch * 1024 + 1024, 0]
        
        loss = torch.nn.functional.mse_loss(predictions[..., 0], max_arr_0[None, 0] * targets.unsqueeze(dim=1) / max_arr_0[None, 1])
        loss_2 = torch.nn.functional.mse_loss(predictions[..., 1], max_arr_0[None, 0] * targets.unsqueeze(dim=1) / max_arr_0[None, 1])
        (loss + loss_2).backward()
        optimizer.step()

        if epoch % 100 == 0:
            pbar.set_postfix({"loss:": loss.detach().item(), "loss2:": loss_2.detach().item()})

  0%|          | 0/9 [00:00<?, ?it/s]

/var/folders/h9/qpwsxl2n43zd4mlpdbkmfs0m0000gp/T/ipykernel_71719/1477925947.py:23: UserWarning: Using a target size (torch.Size([1024, 1])) that is different to the input size (torch.Size([1024])). This will likely lead to incorrect results due to broadcasting. Please ensure they have the same size.
  loss = torch.nn.functional.mse_loss(predictions[..., 0], max_arr_0[None, 0] * targets.unsqueeze(dim=1) / max_arr_0[None, 1])
/var/folders/h9/qpwsxl2n43zd4mlpdbkmfs0m0000gp/T/ipykernel_71719/1477925947.py:24: UserWarning: Using a target size (torch.Size([1024, 1])) that is different to the input size (torch.Size([1024])). This will likely lead to incorrect results due to broadcasting. Please ensure they have the same size.
  loss_2 = torch.nn.functional.mse_loss(predictions[..., 1], max_arr_0[None, 0] * targets.unsqueeze(dim=1) / max_arr_0[None, 1])


  0%|          | 0/9 [00:00<?, ?it/s]

  0%|          | 0/9 [00:00<?, ?it/s]

  0%|          | 0/9 [00:00<?, ?it/s]

  0%|          | 0/9 [00:00<?, ?it/s]

  0%|          | 0/9 [00:00<?, ?it/s]

  0%|          | 0/9 [00:00<?, ?it/s]

  0%|          | 0/9 [00:00<?, ?it/s]

  0%|          | 0/9 [00:00<?, ?it/s]

  0%|          | 0/9 [00:00<?, ?it/s]

In [167]:
test(arr[:10].float()/max_arr_0[None, :])

tensor([[1.5036, 1.5704],
        [1.4861, 1.5599],
        [1.5015, 1.5619],
        [1.4526, 1.5587],
        [1.4954, 1.5665],
        [1.2817, 1.5575],
        [1.4582, 1.5601],
        [1.4991, 1.5638],
        [1.5045, 1.5614],
        [1.4858, 1.5598]], grad_fn=<DivBackward0>)

In [168]:
arr[:10, 1], arr[:10, 0], max_arr_0[None, 0] * arr[:10, 1] / (max_arr_0[None, 1] * arr[:10, 0])

(tensor([4.3454, 8.9071, 7.8980, 9.2831, 5.1913, 9.9664, 7.9499, 6.4969, 8.5518,
         8.9961]),
 tensor([600., 500., 700., 300., 500., 100., 300., 600., 800., 500.]),
 tensor([0.6519, 1.6034, 1.0155, 2.7851, 0.9345, 8.9704, 2.3851, 0.9746, 0.9621,
         1.6194]))

In [152]:
test.model.weight

Parameter containing:
tensor([[ 0.0177, -0.0006],
        [ 0.0002,  0.0181]], requires_grad=True)